# Assignment Part 4 — Data Visualization & Machine Learning
## Theme: Student Performance Analysis & Prediction

We load a small student dataset, explore it with pandas, visualise patterns
using Matplotlib and Seaborn, then train a Logistic Regression classifier
to predict whether a student will pass or fail.


In [1]:
# =====================================================================
# Imports — bring in everything we'll need up front
# =====================================================================
# pandas  : tabular data handling
# matplotlib : low-level plotting (Tasks 2 & 3)
# seaborn : higher-level statistical plots (Task 3)
# sklearn : machine learning pipeline (Task 4)
# =====================================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Use a non-interactive backend so plots render in the notebook
matplotlib.use("Agg")

print("All libraries loaded successfully.")


All libraries loaded successfully.


## Task 1 — Data Exploration with Pandas

In [2]:
# =====================================================================
# TASK 1 — Data Exploration with Pandas
# =====================================================================
# We load the CSV and get a feel for the data before touching it.
# Step by step: look at shape, types, stats, pass/fail split,
# subject averages split by outcome, and the top student.
# =====================================================================

# Load the dataset
var_Df = pd.read_csv("students.csv")

# Step 1: First 5 rows
print("=== First 5 rows ===")
print(var_Df.head())


=== First 5 rows ===
      name  math  science  english  history  pe  attendance_pct  \
0    Alice    88       92       76       80  95              92   
1      Bob    42       55       48       50  60              65   
2  Charlie    75       70       80       68  88              85   
3    Diana    95       98       91       89  97              98   
4      Eve    38       42       50       45  55              58   

   study_hours_per_day  passed  
0                  4.5       1  
1                  1.2       0  
2                  3.0       1  
3                  6.0       1  
4                  0.8       0  


In [3]:
# Step 2: Shape and data types
print(f"\n=== Shape ===")
print(f"Rows: {var_Df.shape[0]}, Columns: {var_Df.shape[1]}")
print("\n=== Data Types ===")
print(var_Df.dtypes)



=== Shape ===
Rows: 15, Columns: 9

=== Data Types ===
name                    object
math                     int64
science                  int64
english                  int64
history                  int64
pe                       int64
attendance_pct           int64
study_hours_per_day    float64
passed                   int64
dtype: object


In [4]:
# Step 3: Summary statistics for all numeric columns
print("\n=== Summary Statistics ===")
print(var_Df.describe().round(2))



=== Summary Statistics ===
        math  science  english  history     pe  attendance_pct  \
count  15.00    15.00    15.00    15.00  15.00           15.00   
mean   65.00    66.73    66.20    63.40  74.80           75.80   
std    20.06    18.97    17.77    16.94  16.66           14.72   
min    30.00    35.00    40.00    28.00  45.00           50.00   
25%    51.50    53.50    49.00    53.50  61.00           63.50   
50%    65.00    65.00    70.00    62.00  75.00           78.00   
75%    80.00    77.00    81.00    73.50  89.00           86.50   
max    95.00    98.00    91.00    92.00  97.00           98.00   

       study_hours_per_day  passed  
count                15.00   15.00  
mean                  2.89    0.60  
std                   1.66    0.51  
min                   0.50    0.00  
25%                   1.65    0.00  
50%                   2.80    1.00  
75%                   3.90    1.00  
max                   6.00    1.00  


In [5]:
# Step 4: Count of passed vs failed students
# passed = 1 means Pass, passed = 0 means Fail
print("\n=== Pass / Fail Count ===")
var_PassFailCounts = var_Df["passed"].value_counts()
print(f"  Pass (1): {var_PassFailCounts.get(1, 0)} students")
print(f"  Fail (0): {var_PassFailCounts.get(0, 0)} students")



=== Pass / Fail Count ===
  Pass (1): 9 students
  Fail (0): 6 students


In [6]:
# Step 5: Average score per subject, split by passed/failed
# This helps us see which subjects separate the two groups the most
var_SubjectCols = ["math", "science", "english", "history", "pe"]

var_PassingAvg = var_Df[var_Df["passed"] == 1][var_SubjectCols].mean().round(2)
var_FailingAvg = var_Df[var_Df["passed"] == 0][var_SubjectCols].mean().round(2)

print("\n=== Average Scores: Passing Students ===")
print(var_PassingAvg.to_string())

print("\n=== Average Scores: Failing Students ===")
print(var_FailingAvg.to_string())



=== Average Scores: Passing Students ===
math       78.22
science    78.56
english    79.11
history    73.44
pe         86.00

=== Average Scores: Failing Students ===
math       45.17
science    49.00
english    46.83
history    48.33
pe         58.00


In [7]:
# Step 6: Student with the highest overall average across all 5 subjects
# We compute a temporary average column and find the max
var_Df["temp_avg"] = var_Df[var_SubjectCols].mean(axis=1)
var_TopStudentIdx = var_Df["temp_avg"].idxmax()
var_TopStudent    = var_Df.loc[var_TopStudentIdx]

print("\n=== Top Student (highest subject average) ===")
print(f"  Name    : {var_TopStudent['name']}")
print(f"  Average : {var_TopStudent['temp_avg']:.2f}")

# Drop the temporary column — we don't need it cluttering the DataFrame
var_Df = var_Df.drop(columns=["temp_avg"])



=== Top Student (highest subject average) ===
  Name    : Diana
  Average : 94.00


## Task 2 — Data Visualization with Matplotlib

We create 5 plots, each saved as a `.png` file and displayed inline.
Before we start, we add an `avg_score` column (mean of the 5 subjects per student).


In [8]:
# =====================================================================
# TASK 2 — Data Visualization with Matplotlib
# =====================================================================
# Add avg_score column as instructed — we use it in the scatter plot
# =====================================================================

var_SubjectCols  = ["math", "science", "english", "history", "pe"]
var_Df["avg_score"] = var_Df[var_SubjectCols].mean(axis=1)

print("avg_score column added. Preview:")
print(var_Df[["name", "avg_score"]].to_string(index=False))


avg_score column added. Preview:
   name  avg_score
  Alice       86.2
    Bob       51.0
Charlie       76.2
  Diana       94.0
    Eve       46.0
  Frank       65.0
  Grace       52.2
  Henry       82.6
   Iris       71.0
   Jack       35.6
  Karen       66.4
   Liam       51.4
    Mia       92.2
   Noah       60.6
 Olivia       78.0


In [9]:
# =====================================================================
# Plot 1 — Bar Chart: Average score per subject across all students
# =====================================================================
# One bar per subject — shows which subjects the class performs well/poorly in

var_SubjectAvgs = var_Df[var_SubjectCols].mean()

fig, ax = plt.subplots(figsize=(8, 5))

# Use a colour per bar so they're visually distinct
var_BarColors = ["steelblue", "coral", "mediumseagreen", "mediumpurple", "goldenrod"]
ax.bar(var_SubjectCols, var_SubjectAvgs, color=var_BarColors, edgecolor="black", width=0.5)

ax.set_title("Average Score Per Subject (All Students)", fontsize=14, fontweight="bold")
ax.set_xlabel("Subject", fontsize=12)
ax.set_ylabel("Average Score", fontsize=12)
ax.set_ylim(0, 100)

# Add value labels on top of each bar
for var_Bar in ax.patches:
    ax.text(
        var_Bar.get_x() + var_Bar.get_width() / 2,
        var_Bar.get_height() + 1,
        f"{var_Bar.get_height():.1f}",
        ha="center", va="bottom", fontsize=10
    )

plt.tight_layout()
plt.savefig("plot1_bar.png", dpi=100)
plt.show()
print("plot1_bar.png saved.")


plot1_bar.png saved.


In [10]:
# =====================================================================
# Plot 2 — Histogram: Distribution of math scores
# =====================================================================
# 5 bins, with a vertical dashed line at the mean math score

var_MathMean = var_Df["math"].mean()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(var_Df["math"], bins=5, color="steelblue", edgecolor="black", alpha=0.8)

# Dashed line at the mean — helps readers see how scores cluster
ax.axvline(var_MathMean, color="red", linestyle="--", linewidth=2, label=f"Mean = {var_MathMean:.1f}")

ax.set_title("Distribution of Math Scores", fontsize=14, fontweight="bold")
ax.set_xlabel("Math Score", fontsize=12)
ax.set_ylabel("Number of Students", fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig("plot2_histogram.png", dpi=100)
plt.show()
print("plot2_histogram.png saved.")


plot2_histogram.png saved.


In [11]:
# =====================================================================
# Plot 3 — Scatter Plot: Study hours vs Average score, coloured by pass/fail
# =====================================================================
# We plot the two groups separately so we can give them different colours
# and add them individually to the legend

var_PassDf = var_Df[var_Df["passed"] == 1]
var_FailDf = var_Df[var_Df["passed"] == 0]

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(var_PassDf["study_hours_per_day"], var_PassDf["avg_score"],
           color="green", label="Pass", s=80, edgecolors="black", zorder=5)

ax.scatter(var_FailDf["study_hours_per_day"], var_FailDf["avg_score"],
           color="red", label="Fail", s=80, edgecolors="black", zorder=5)

ax.set_title("Study Hours vs Average Score (Pass / Fail)", fontsize=14, fontweight="bold")
ax.set_xlabel("Study Hours Per Day", fontsize=12)
ax.set_ylabel("Average Score", fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig("plot3_scatter.png", dpi=100)
plt.show()
print("plot3_scatter.png saved.")


plot3_scatter.png saved.


In [12]:
# =====================================================================
# Plot 4 — Box Plot: Attendance % for Pass vs Fail students
# =====================================================================
# Side-by-side boxes make it easy to see the spread and median for each group

var_PassAttendance = var_Df[var_Df["passed"] == 1]["attendance_pct"].tolist()
var_FailAttendance = var_Df[var_Df["passed"] == 0]["attendance_pct"].tolist()

fig, ax = plt.subplots(figsize=(7, 5))
bp = ax.boxplot(
    [var_PassAttendance, var_FailAttendance],
    labels=["Pass", "Fail"],
    patch_artist=True,       # fill boxes with colour
    notch=False
)

# Colour the boxes differently so they stand out
bp["boxes"][0].set_facecolor("lightgreen")
bp["boxes"][1].set_facecolor("lightcoral")

ax.set_title("Attendance % Distribution: Pass vs Fail", fontsize=14, fontweight="bold")
ax.set_xlabel("Result", fontsize=12)
ax.set_ylabel("Attendance %", fontsize=12)

plt.tight_layout()
plt.savefig("plot4_boxplot.png", dpi=100)
plt.show()
print("plot4_boxplot.png saved.")


plot4_boxplot.png saved.


/tmp/ipykernel_37478/3042898193.py:10: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


In [13]:
# =====================================================================
# Plot 5 — Line Plot: Math and Science scores for every student
# =====================================================================
# Two lines (one per subject) over student names on the x-axis
# Rotated labels prevent overlap with 15 names

var_StudentNames = var_Df["name"].tolist()
var_MathScores   = var_Df["math"].tolist()
var_SciScores    = var_Df["science"].tolist()

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(var_StudentNames, var_MathScores,
        marker="o", linestyle="-",  color="steelblue", label="Math",    linewidth=2)
ax.plot(var_StudentNames, var_SciScores,
        marker="s", linestyle="--", color="coral",     label="Science", linewidth=2)

ax.set_title("Math and Science Scores Per Student", fontsize=14, fontweight="bold")
ax.set_xlabel("Student Name", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.legend(fontsize=11)
ax.set_ylim(0, 110)
plt.xticks(rotation=45, ha="right", fontsize=9)

plt.tight_layout()
plt.savefig("plot5_line.png", dpi=100)
plt.show()
print("plot5_line.png saved.")


plot5_line.png saved.


## Task 3 — Data Visualization with Seaborn

In [14]:
# =====================================================================
# TASK 3 — Seaborn Visualizations
# =====================================================================

# Plot 6 — Seaborn bar plot: Average math and science scores by pass/fail
# We use two subplots side by side, one sns.barplot per subject
# This is simpler than reshaping the data and gives a clean comparison.
# =====================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

sns.barplot(data=var_Df, x="passed", y="math",    ax=ax1, palette=["lightcoral", "steelblue"])
ax1.set_title("Avg Math Score by Result",    fontsize=12, fontweight="bold")
ax1.set_xlabel("Passed (0 = Fail, 1 = Pass)")
ax1.set_ylabel("Math Score")

sns.barplot(data=var_Df, x="passed", y="science", ax=ax2, palette=["lightcoral", "steelblue"])
ax2.set_title("Avg Science Score by Result", fontsize=12, fontweight="bold")
ax2.set_xlabel("Passed (0 = Fail, 1 = Pass)")
ax2.set_ylabel("Science Score")

plt.suptitle("Math & Science Averages: Pass vs Fail", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("plot6_seaborn_bar.png", dpi=100, bbox_inches="tight")
plt.show()
print("plot6_seaborn_bar.png saved.")


/tmp/ipykernel_37478/353814560.py:12: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=var_Df, x="passed", y="math",    ax=ax1, palette=["lightcoral", "steelblue"])


plot6_seaborn_bar.png saved.


/tmp/ipykernel_37478/353814560.py:17: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=var_Df, x="passed", y="science", ax=ax2, palette=["lightcoral", "steelblue"])


In [15]:
# =====================================================================
# Plot 7 — Seaborn scatter with regression lines:
#           Attendance % vs Average Score, coloured by Pass/Fail
# =====================================================================
# We call sns.regplot twice — once for each group — on the same axes.
# This draws the scatter points AND fits a regression line per group,
# showing whether attendance predicts average score differently for
# passing vs failing students.

fig, ax = plt.subplots(figsize=(9, 5))

sns.regplot(
    data=var_Df[var_Df["passed"] == 1],
    x="attendance_pct", y="avg_score",
    ax=ax, label="Pass",
    color="green", scatter_kws={"s": 60, "edgecolors": "black"},
    line_kws={"linewidth": 2}
)

sns.regplot(
    data=var_Df[var_Df["passed"] == 0],
    x="attendance_pct", y="avg_score",
    ax=ax, label="Fail",
    color="red", scatter_kws={"s": 60, "edgecolors": "black"},
    line_kws={"linewidth": 2}
)

ax.set_title("Attendance % vs Average Score (with Regression Lines)", fontsize=13, fontweight="bold")
ax.set_xlabel("Attendance %", fontsize=12)
ax.set_ylabel("Average Score", fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig("plot7_seaborn_scatter.png", dpi=100)
plt.show()
print("plot7_seaborn_scatter.png saved.")


plot7_seaborn_scatter.png saved.


In [16]:
# =====================================================================
# Seaborn vs Matplotlib — comparison comment
# =====================================================================
# Seaborn made statistical plots (bar with confidence intervals,
# scatter with regression lines) much faster to write — sns.barplot
# and sns.regplot each did in one call what would have taken several
# lines of matplotlib code (computing means, fitting lines manually).
# Matplotlib gave finer control over layout, colours, and annotations
# (e.g., labelling bars or adding custom dashed lines), so it's better
# when you need precise customisation. For exploratory analysis,
# Seaborn is quicker; for publication-ready or interactive charts,
# Matplotlib's flexibility pays off.
print("Seaborn vs Matplotlib comparison noted above as a code comment.")


Seaborn vs Matplotlib comparison noted above as a code comment.


## Task 4 — Machine Learning with scikit-learn

We train a Logistic Regression classifier to predict Pass (1) or Fail (0)
based on subject scores, attendance, and study hours.


In [17]:
# =====================================================================
# TASK 4 — Machine Learning with scikit-learn
# =====================================================================
# Step 1: Prepare Data
# =====================================================================
# X = feature columns (everything except name and passed)
# y = target (passed column)
# We keep name in var_Df so we can look it up for the per-student report.
# =====================================================================

var_FeatureCols = ["math", "science", "english", "history", "pe",
                   "attendance_pct", "study_hours_per_day"]

var_X = var_Df[var_FeatureCols]   # features only — no name, no passed
var_y = var_Df["passed"]          # target column

print("Features shape:", var_X.shape)
print("Target shape:  ", var_y.shape)


Features shape: (15, 7)
Target shape:   (15,)


In [18]:
# Step 2: Train / Test split — 80% training, 20% test
# random_state=42 makes the split reproducible
var_X_Train, var_X_Test, var_y_Train, var_y_Test = train_test_split(
    var_X, var_y, test_size=0.2, random_state=42
)

print(f"Training samples : {len(var_X_Train)}")
print(f"Test samples     : {len(var_X_Test)}")


Training samples : 12
Test samples     : 3


In [19]:
# Step 3: Scale features using StandardScaler
# We fit ONLY on training data so test data has no influence on the scaling —
# this prevents data leakage.
var_Scaler      = StandardScaler()
var_XTrain_Scaled = var_Scaler.fit_transform(var_X_Train)   # fit + transform training data
var_XTest_Scaled  = var_Scaler.transform(var_X_Test)         # transform test data using training stats

print("Features scaled. Training set preview (first row):")
print(var_XTrain_Scaled[0].round(3))


Features scaled. Training set preview (first row):
[-0.55  -0.402 -0.908 -0.405 -0.658 -0.621 -0.679]


In [20]:
# =====================================================================
# Step 4: Train a Logistic Regression model
# =====================================================================
# Logistic Regression works well for binary classification (pass/fail).
# max_iter=500 gives the solver enough iterations to converge on
# this small dataset.
# =====================================================================

var_Model = LogisticRegression(max_iter=500, random_state=42)
var_Model.fit(var_XTrain_Scaled, var_y_Train)

# Training accuracy — how well the model fits the data it learned from
var_TrainAccuracy = var_Model.score(var_XTrain_Scaled, var_y_Train)
print(f"Training Accuracy : {var_TrainAccuracy * 100:.1f}%")


Training Accuracy : 100.0%


In [21]:
# =====================================================================
# Step 5: Evaluate on the test set
# =====================================================================
# With only 3 test samples (15 * 20% = 3), accuracy will be 0/33/66/100%.
# The goal is to understand the workflow, not to maximise accuracy.
# =====================================================================

var_y_Pred       = var_Model.predict(var_XTest_Scaled)
var_TestAccuracy = accuracy_score(var_y_Test, var_y_Pred)
print(f"Test Accuracy : {var_TestAccuracy * 100:.1f}%")

# Per-student breakdown — show name, actual, predicted, correct/wrong
# var_X_Test retains its original DataFrame index, so we can use
# df.loc[X_test.index, 'name'] to get the matching student names.
print("\nPer-student predictions:")
var_TestNames = var_Df.loc[var_X_Test.index, "name"].tolist()

for var_Idx, (var_Name, var_Actual, var_Predicted) in enumerate(
        zip(var_TestNames, var_y_Test, var_y_Pred)):
    var_Actual_Label    = "Pass" if var_Actual    == 1 else "Fail"
    var_Predicted_Label = "Pass" if var_Predicted == 1 else "Fail"
    var_Correct         = "✅ correct" if var_Actual == var_Predicted else "❌ wrong"
    print(f"  {var_Name:<10}  Actual: {var_Actual_Label:<5}  Predicted: {var_Predicted_Label:<5}  {var_Correct}")


Test Accuracy : 100.0%

Per-student predictions:
  Jack        Actual: Fail   Predicted: Fail   ✅ correct
  Liam        Actual: Fail   Predicted: Fail   ✅ correct
  Alice       Actual: Pass   Predicted: Pass   ✅ correct


In [22]:
# =====================================================================
# Step 6: Feature Importance (model coefficients)
# =====================================================================
# model.coef_[0] gives one coefficient per feature.
# A positive value means the feature pushes the prediction toward Pass;
# a negative value pushes toward Fail.
# We sort by absolute value so the most impactful features appear first.
# =====================================================================

var_Coefficients = var_Model.coef_[0]
var_CoefPairs    = list(zip(var_FeatureCols, var_Coefficients))

# Sort by absolute value descending — biggest influence first
var_CoefPairs.sort(key=lambda pair: abs(pair[1]), reverse=True)

print("Feature Coefficients (sorted by impact):")
for var_Feature, var_Coef in var_CoefPairs:
    var_Direction = "→ Pass" if var_Coef > 0 else "→ Fail"
    print(f"  {var_Feature:<25} {var_Coef:+.4f}   {var_Direction}")


Feature Coefficients (sorted by impact):
  english                   +0.8125   → Pass
  attendance_pct            +0.5219   → Pass
  study_hours_per_day       +0.4844   → Pass
  pe                        +0.4750   → Pass
  math                      +0.4379   → Pass
  science                   +0.3230   → Pass
  history                   +0.2629   → Pass


In [23]:
# =====================================================================
# Plot 8 — Horizontal bar chart of feature coefficients
# =====================================================================
# Green bars = positive coefficient (pushes toward Pass)
# Red bars   = negative coefficient (pushes toward Fail)

var_FeatNames = [pair[0] for pair in var_CoefPairs]
var_CoefVals  = [pair[1] for pair in var_CoefPairs]
var_BarColours = ["green" if c > 0 else "red" for c in var_CoefVals]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(var_FeatNames, var_CoefVals, color=var_BarColours, edgecolor="black")

ax.axvline(0, color="black", linewidth=1)   # centre line at 0
ax.set_title("Logistic Regression — Feature Coefficients", fontsize=14, fontweight="bold")
ax.set_xlabel("Coefficient Value (positive = toward Pass, negative = toward Fail)", fontsize=11)
ax.set_ylabel("Feature", fontsize=12)

# Add value labels at the end of each bar
for var_Bar, var_Val in zip(ax.patches, var_CoefVals):
    var_XPos = var_Bar.get_width() + (0.02 if var_Val >= 0 else -0.02)
    var_HA   = "left" if var_Val >= 0 else "right"
    ax.text(var_XPos, var_Bar.get_y() + var_Bar.get_height() / 2,
            f"{var_Val:+.3f}", va="center", ha=var_HA, fontsize=9)

plt.tight_layout()
plt.savefig("plot8_feature_importance.png", dpi=100)
plt.show()
print("plot8_feature_importance.png saved.")


plot8_feature_importance.png saved.
